**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# Triple-Regime (A/B/C) Features for MRvF Deep Learning

## Extension of the Dual-Regime (A/B) model

The dual-regime model added two features — R2\*_A and R2\*_B — derived from the
FID (Part A) and rephasing (Part B) segments of the GESFIDE signal.

This notebook adds a **third feature from Part C (post-SE decay)**:

| Part | Echoes | Physical model | R2* encodes |
|------|--------|----------------|-------------|
| A (FID)       | 0–13   | `S_A(t) = S0 · exp(−R2*_A · t)` | R2 + R2'  |
| B (rephasing) | 14–29  | `S_B(t) ∝ exp(−R2*_B · t)` | R2 − R2' (can be negative) |
| C (post-SE)   | 30–39  | `S_C(t) = S_SE · exp(−R2*_C · (t − T_SE))` | R2 + R2' again |

**Why Part C adds information beyond Part A:**
- Part C decays from the spin-echo amplitude, so its starting point is purely R2-weighted
  (R2' has fully refocused at the SE). This gives an independent R2 + R2' measurement
  at a different point along the T2 decay curve.
- The ratio `R2*_C / R2*_A` is sensitive to how R2' evolves away from the SE —
  encoding microvascular geometry in a different way to Part A.
- Together, the three features span a richer projection of the (R2, R2') space:
  ```
  R2  ≈ (R2*_A + R2*_B) / 2
  R2' ≈ (R2*_A − R2*_B) / 2
  R2*_C provides an independent cross-check on both
  ```

**Network input**: 40 L2-normalised echoes + R2*_A_scaled + R2*_B_scaled + R2*_C_scaled = **43 dims**  
**Network output**: SO₂, CBV, R, T2 (4 parameters)

Final section: **ablation study comparing 2-feature (A+B) vs 3-feature (A+B+C)**
using the same conditions as the dual-regime ablation.

---

## 1. Imports & GPU

In [ ]:
import os, json, time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── Match plot style of 04_Generate_Figures_T2cond_v5.ipynb ──────────────
plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

# ── Colour palette (matches reference notebook exactly) ──────────────────
C_DM          = '#E65100'   # deep orange  — DM
C_DL_NF       = '#2E7D32'   # deep green   — DL noise-free
C_DL_NOISY    = '#1565C0'   # deep blue    — DL noisy / dual-regime A+B
C_T2COND_BC   = '#6A1B9A'   # deep purple  — triple-regime A+B+C
C_T2COND_ABC  = '#AD1457'   # deep pink    — reserved

# New colour for this model
C_TRIPLE      = C_T2COND_BC   # deep purple for A+B+C

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('CPU only')
print(f'PyTorch {torch.__version__}')

## 2. Configuration

In [ ]:
CONFIG = {
    # --- Data paths ---
    'dict_base_path':     '../subsamples/subsamples_v3',
    'param_path':         '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'noisefree_sig_path': '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'echotimes_path':     '../echotimes.mat',
    'output_dir':         './results/triple_regime_results_v1',
    # Path to saved dual-regime (A+B) checkpoint for ablation comparison
    'dual_regime_ckpt':   './results/dual_regime_results_v1/models/dual_regime_best.pt',
    'dict_key':           'Dico40_save',
    'param_key':          'par_save',

    # --- Parameter space ---
    'param_mins':   np.array([0.0,    0.0025,  1.0e-6,  0.050]),
    'param_maxs':   np.array([1.0,    0.15,   25.0e-6,  0.200]),
    'param_names':  ['SO2', 'CBV', 'R', 'T2'],

    # --- GESFIDE geometry ---
    'n_fid':    14,   # Part A: echoes  0-13
    'n_rephas': 16,   # Part B: echoes 14-29  (spin echo = echo 29)
    'n_postse': 10,   # Part C: echoes 30-39

    # --- Feature scaling ranges (s^-1) ---
    # R2*_A = R2 + R2':  always positive
    'R2starA_min':  2.0,   'R2starA_max': 55.0,
    # R2*_B = R2 - R2':  can be negative
    'R2starB_min': -30.0,  'R2starB_max': 22.0,
    # R2*_C = R2 + R2' (post-SE):  always positive, similar range to A
    'R2starC_min':  2.0,   'R2starC_max': 55.0,

    # --- Training ---
    'snr_levels':    [20, 50, 100, 150],
    'n_samples':     1_600_000,
    'test_frac':     0.15,
    'val_frac':      0.15,
    'batch_size':    16384,
    'epochs':        200,
    'patience':      25,
    'lr':            5e-5,
    'dropout':       0.05,
    'param_weights': [1.0, 12.0, 8.0, 1.0],
}

SE_ECHO = CONFIG['n_fid'] + CONFIG['n_rephas']   # = 30
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'models'), exist_ok=True)
FIG_DIR = os.path.join(CONFIG['output_dir'], 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

PARAM_NAMES = ['SO₂', 'CBV', 'R',   'T2']
PARAM_UNITS = ['(%)', '(%)', '(µm)', '(ms)']
PARAM_SCALE = [100,   100,   1e6,    1000]
PKEYS       = CONFIG['param_names']
SNR_LEVELS  = CONFIG['snr_levels']

print(f'SE_ECHO={SE_ECHO}  Part C: echoes {SE_ECHO}-{SE_ECHO+CONFIG["n_postse"]-1}')
print(f'Input dim: 40 + 3 = 43  (signal + R2*_A + R2*_B + R2*_C)')

## 3. Load echo times

In [ ]:
et_mat       = sio.loadmat(CONFIG['echotimes_path'])
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0

T_A = echo_times_s[:CONFIG['n_fid']]                         # Part A
T_B = echo_times_s[CONFIG['n_fid']:SE_ECHO]                  # Part B
T_C = echo_times_s[SE_ECHO:]                                  # Part C (absolute times from excitation)
T_SE_S = echo_times_s[SE_ECHO - 1]                           # spin echo time
T_C_rel = T_C - T_SE_S                                        # Part C times relative to SE

print(f'Part A: echoes  0-{CONFIG["n_fid"]-1},   t = [{T_A[0]*1e3:.2f}, ..., {T_A[-1]*1e3:.2f}] ms')
print(f'Part B: echoes {CONFIG["n_fid"]}-{SE_ECHO-1},   t = [{T_B[0]*1e3:.2f}, ..., {T_B[-1]*1e3:.2f}] ms')
print(f'Part C: echoes {SE_ECHO}-{SE_ECHO+CONFIG["n_postse"]-1},  t = [{T_C[0]*1e3:.2f}, ..., {T_C[-1]*1e3:.2f}] ms')
print(f'Spin echo at t = {T_SE_S*1e3:.2f} ms')
print(f'Part C relative to SE: [{T_C_rel[0]*1e3:.2f}, ..., {T_C_rel[-1]*1e3:.2f}] ms')

## 4. Utilities

In [ ]:
def load_mat(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found, using "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)

def params_scale(p, mins, maxs):
    return ((p - mins) / (maxs - mins)).astype(np.float32)

def params_inverse(p, mins, maxs):
    return (p * (maxs - mins) + mins).astype(np.float32)

def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    if (~mask).sum():
        print(f'  Filtered {(~mask).sum()} out-of-range entries ({(~mask).mean()*100:.1f}%)')
    return signals[mask], params[mask]

def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    if (~valid).sum(): print(f'  Removed {(~valid).sum()} non-finite entries')
    return signals[valid], params[valid]

def batched_predict(model, x_np, batch_size=4096, dev=device):
    """Run inference in batches to avoid OOM."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch_size):
            xb = torch.tensor(x_np[i:i+batch_size], dtype=torch.float32).to(dev).contiguous()
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)

def save_fig(fig, name):
    for ext in ['pdf', 'png']:
        p = os.path.join(FIG_DIR, f'{name}.{ext}')
        fig.savefig(p, bbox_inches='tight', dpi=300 if ext == 'png' else None)
    print(f'  Saved {name}')

print('Utilities ready.')

## 5. Triple-regime feature computation

**R2\*_A** — Part A OLS (echoes 0–13, absolute t): `R2*_A = −slope_A = R2 + R2'`  
**R2\*_B** — Part B OLS (echoes 14–29, absolute t): `R2*_B = −slope_B = R2 − R2'`  
  → Raw signed; negative when Part B rephases (R2' > R2)  
**R2\*_C** — Part C OLS (echoes 30–39, **time relative to SE**): `R2*_C = −slope_C = R2 + R2'`  
  → Part C starts fresh from the SE; using t_rel keeps it comparable to Part A

In [ ]:
def ols_slope(t_vec, sig_mat):
    """OLS slope of log|S| vs t.  Returns raw slope (N,)."""
    log_s = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t     = t_vec.astype(np.float64)
    t_c   = t - t.mean()
    log_sm = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_sm * t_c[None, :]).sum(axis=1) / (t_c ** 2).sum()


def compute_triple_regime_features(sig_raw, config):
    """
    Compute R2*_A, R2*_B, R2*_C from un-normalised signal.

    Returns
    -------
    R2starA : (N,) s^-1,  >= 0 (lower-clipped)
    R2starB : (N,) s^-1,  raw signed (can be negative)
    R2starC : (N,) s^-1,  >= 0 (lower-clipped), fitted on t_rel from SE
    """
    n_fid   = config['n_fid']
    se_echo = n_fid + config['n_rephas']   # = 30
    r2_lb   = 1.0 / config['param_maxs'][3]   # ~5 s^-1

    # Part A
    slope_A = ols_slope(T_A, sig_raw[:, :n_fid])
    R2starA = np.maximum(-slope_A, r2_lb).astype(np.float32)

    # Part B — raw signed, no clip
    slope_B = ols_slope(T_B, sig_raw[:, n_fid:se_echo])
    R2starB = (-slope_B).astype(np.float32)

    # Part C — fit on TIME RELATIVE TO SE so intercept = log(S_SE)
    slope_C = ols_slope(T_C_rel, sig_raw[:, se_echo:])
    R2starC = np.maximum(-slope_C, r2_lb).astype(np.float32)

    return R2starA, R2starB, R2starC


def scale_triple_features(R2starA, R2starB, R2starC, config):
    """Min-max scale all three regime features to [0, 1]."""
    def scale(x, lo, hi):
        return np.clip((x - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)
    feat_A = scale(R2starA, config['R2starA_min'], config['R2starA_max'])
    feat_B = scale(R2starB, config['R2starB_min'], config['R2starB_max'])
    feat_C = scale(R2starC, config['R2starC_min'], config['R2starC_max'])
    return feat_A, feat_B, feat_C


print('Triple-regime feature functions ready.')

## 6. Diagnostic — noise-free validation of all three features

In [ ]:
print('Loading noise-free signals...')
nf_sig    = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
nf_params = load_mat(CONFIG['param_path'],         CONFIG['param_key'])[:, :4]
nf_sig, nf_params = filter_param_range(nf_sig, nf_params, CONFIG['param_mins'], CONFIG['param_maxs'])
nf_sig, nf_params = clean_data(nf_sig, nf_params)

rng   = np.random.default_rng(42)
idx_d = rng.choice(len(nf_sig), 50_000, replace=False)
sig_d = nf_sig[idx_d];  par_d = nf_params[idx_d]

R2starA_d, R2starB_d, R2starC_d = compute_triple_regime_features(sig_d, CONFIG)
T2_true_d = par_d[:, 3] * 1000.0
R2_true_d = 1.0 / par_d[:, 3]

R2_comb_d = (R2starA_d + R2starB_d) / 2.0
R2p_d     = (R2starA_d - R2starB_d) / 2.0

print(f'\nNoise-free (N={len(sig_d):,})')
print(f'  R2*_A: [{R2starA_d.min():.1f}, {R2starA_d.max():.1f}] s^-1  mean={R2starA_d.mean():.1f}')
print(f'  R2*_B: [{R2starB_d.min():.1f}, {R2starB_d.max():.1f}] s^-1  (negative = Part B rephases)')
print(f'  R2*_C: [{R2starC_d.min():.1f}, {R2starC_d.max():.1f}] s^-1  mean={R2starC_d.mean():.1f}')
print(f'  Fraction R2*_B < 0: {(R2starB_d < 0).mean()*100:.1f}%')
print(f'  R2 from (A+B)/2:  RMSE={np.sqrt(np.mean((R2_comb_d-R2_true_d)**2)):.4f} s^-1')

# Scatter: R2*_A vs R2*_C (should be correlated — both encode R2+R2' at different points)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].scatter(R2starA_d[:3000], R2starC_d[:3000], s=2, alpha=0.3, c=C_DL_NOISY)
lo = min(R2starA_d.min(), R2starC_d.min())
hi = max(R2starA_d.max(), R2starC_d.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.0)
axes[0].set_xlabel('R2*_A (s⁻¹)', fontsize=9); axes[0].set_ylabel('R2*_C (s⁻¹)', fontsize=9)
axes[0].set_title('R2*_A vs R2*_C', fontsize=11, fontweight='bold')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].hist(R2starB_d, bins=80, color=C_T2COND_BC, edgecolor='none')
axes[1].axvline(0, color='r', lw=1.5, label='R2*_B = 0')
axes[1].set_xlabel('R2*_B (s⁻¹)', fontsize=9); axes[1].set_ylabel('Count', fontsize=9)
axes[1].set_title('R2*_B distribution', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

T2_A = np.clip(1000.0 / R2starA_d, 0, 500)
T2_C = np.clip(1000.0 / R2starC_d, 0, 500)
axes[2].scatter(T2_true_d[:3000], T2_C[:3000], s=2, alpha=0.3, c=C_T2COND_BC, label='T2_C proxy')
axes[2].scatter(T2_true_d[:3000], T2_A[:3000], s=2, alpha=0.15, c=C_DL_NOISY, label='T2_A proxy')
axes[2].plot([50,200],[50,200],'r--', lw=1.0)
axes[2].set_xlabel('True T2 (ms)', fontsize=9); axes[2].set_ylabel('T2 proxy (ms)', fontsize=9)
axes[2].set_title('T2 proxies from A and C', fontsize=11, fontweight='bold')
axes[2].legend(fontsize=8, markerscale=3)
axes[2].spines['top'].set_visible(False); axes[2].spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'diagnostic_triple_features')
plt.show()

## 7. Dataset preparation — 43-dim input

In [ ]:
def prepare_triple_regime_dataset(config):
    print('=' * 65)
    print('BUILDING TRIPLE-REGIME MIXED-SNR DATASET (43-dim input)')
    print('=' * 65)

    params_raw = load_mat(config['param_path'], config['param_key'])[:, :4]
    all_x, all_y = [], []

    for snr in config['snr_levels']:
        print(f'\n  SNR={snr}...')
        sig_path = os.path.join(config['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
        sig_raw  = load_mat(sig_path, config['dict_key'])
        sig_i, par_i = filter_param_range(
            sig_raw, params_raw.copy(), config['param_mins'], config['param_maxs'])

        # Step 1: compute all three regime features from RAW signal
        R2A, R2B, R2C  = compute_triple_regime_features(sig_i, config)
        fA, fB, fC     = scale_triple_features(R2A, R2B, R2C, config)

        # Step 2: L2-normalise signal
        sig_norm, par_i = clean_data(euclidean_norm(sig_i), par_i)
        n = len(par_i)
        fA, fB, fC = fA[:n], fB[:n], fC[:n]

        # Step 3: assemble 43-dim input
        x_i = np.concatenate([
            sig_norm,        # (N, 40)
            fA[:, None],     # (N,  1)  R2*_A
            fB[:, None],     # (N,  1)  R2*_B
            fC[:, None],     # (N,  1)  R2*_C
        ], axis=1)           # (N, 43)

        y_i = params_scale(par_i[:, :4], config['param_mins'][:4], config['param_maxs'][:4])

        all_x.append(x_i); all_y.append(y_i)
        print(f'    {n:,} samples | fA [{fA.min():.2f},{fA.max():.2f}] '
              f'fB [{fB.min():.2f},{fB.max():.2f}] '
              f'fC [{fC.min():.2f},{fC.max():.2f}]')

    X = np.vstack(all_x); Y = np.vstack(all_y)
    if len(X) > config['n_samples']:
        idx = np.random.choice(len(X), config['n_samples'], replace=False)
        X, Y = X[idx], Y[idx]

    print(f'\nTotal: {len(X):,}  input_dim={X.shape[1]}  output_dim={Y.shape[1]}')
    x_tv, x_test, y_tv, y_test = train_test_split(X, Y, test_size=config['test_frac'], random_state=42)
    x_train, x_val, y_train, y_val = train_test_split(x_tv, y_tv, test_size=config['val_frac'], random_state=42)
    y_test_raw = params_inverse(y_test, config['param_mins'][:4], config['param_maxs'][:4])
    print(f'  Train={len(x_train):,}  Val={len(x_val):,}  Test={len(x_test):,}')
    return x_train, x_val, x_test, y_train, y_val, y_test, y_test_raw


(x_train, x_val, x_test,
 y_train, y_val,
 y_test,  y_test_raw) = prepare_triple_regime_dataset(CONFIG)

print(f'\nInput dim:  {x_train.shape[1]}  (expected 43)')
print(f'feat_A range: [{x_train[:,40].min():.3f},{x_train[:,40].max():.3f}]')
print(f'feat_B range: [{x_train[:,41].min():.3f},{x_train[:,41].max():.3f}]')
print(f'feat_C range: [{x_train[:,42].min():.3f},{x_train[:,42].max():.3f}]')

## 8. Model architecture — 43-dim input, 3-scalar FiLM conditioning

In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)


class FiLMLayer(nn.Module):
    """Feature-wise Linear Modulation: y = gamma(cond) * x + beta(cond)"""
    def __init__(self, feature_dim, cond_in=3, cond_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_in, cond_hidden), nn.ReLU(),
            nn.Linear(cond_hidden, 2 * feature_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        b = torch.zeros(2 * feature_dim)
        b[:feature_dim] = 1.0   # gamma starts at 1
        self.net[-1].bias.data.copy_(b)
        self.feature_dim = feature_dim

    def forward(self, x, cond):
        params = self.net(cond)
        gamma, beta = params[:, :self.feature_dim], params[:, self.feature_dim:]
        return gamma * x + beta


class TripleRegimeModel(nn.Module):
    """
    Conv1D backbone + FiLM conditioning on three regime scalars (R2*_A, R2*_B, R2*_C).

    Input:  (B, 43) = 40-echo signal + feat_A + feat_B + feat_C
    Output: (B,  4) = SO2, CBV, R, T2 in [0,1]
    """
    def __init__(self, n_outputs=4, dropout=0.05, film_cond_hidden=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32,  kernel_size=7, padding=3),
            nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64,  kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
        )  # 40 → 20 → 10 → 5,  conv_out = 256 * 5 = 1280

        self.fc1   = nn.Linear(1280, 512); self.bn1 = nn.BatchNorm1d(512)
        self.film1 = FiLMLayer(512, cond_in=3, cond_hidden=film_cond_hidden)
        self.fc2   = nn.Linear(512,  256); self.bn2 = nn.BatchNorm1d(256)
        self.film2 = FiLMLayer(256, cond_in=3, cond_hidden=film_cond_hidden)
        self.fc3   = nn.Linear(256,  128); self.bn3 = nn.BatchNorm1d(128)
        self.film3 = FiLMLayer(128, cond_in=3, cond_hidden=film_cond_hidden)

        self.fc_out  = nn.Linear(128, n_outputs)
        self.out_act = Clamp01()
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU()

    def forward(self, x):
        echo = x[:, :40].unsqueeze(1)   # (B, 1, 40)
        cond = x[:, 40:43]              # (B, 3)  [feat_A, feat_B, feat_C]
        c = self.conv(echo).flatten(1)  # (B, 1280)
        h = self.drop(self.relu(self.bn1(self.film1(self.fc1(c), cond))))
        h = self.drop(self.relu(self.bn2(self.film2(self.fc2(h), cond))))
        h = self.drop(self.relu(self.bn3(self.film3(self.fc3(h), cond))))
        return self.out_act(self.fc_out(h))


_m = TripleRegimeModel().to(device)
_x = torch.randn(8, 43).to(device)
print(f'Output shape: {_m(_x).shape}   (expected [8, 4])')
print(f'Parameters:   {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x

## 9. Loss & training loop

In [ ]:
class WeightedMAELoss(nn.Module):
    def __init__(self, weights):
        super().__init__()
        self.register_buffer('w', torch.tensor(weights, dtype=torch.float32))
    def forward(self, pred, target):
        return (torch.abs(pred - target) * self.w.unsqueeze(0)).sum(dim=1).mean()


def train_model(model, x_train, y_train, x_val, y_val, config, ckpt_name='triple_regime_best.pt'):
    criterion = WeightedMAELoss(config['param_weights']).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=8, factor=0.5, min_lr=1e-7, verbose=True)

    x_t = torch.tensor(x_train, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    x_v = torch.tensor(x_val,   dtype=torch.float32).to(device)
    y_v = torch.tensor(y_val,   dtype=torch.float32).to(device)

    loader    = DataLoader(TensorDataset(x_t, y_t), batch_size=config['batch_size'], shuffle=True)
    best_val  = float('inf'); patience_cnt = 0
    history   = {'train_loss': [], 'val_loss': []}
    ckpt_path = os.path.join(config['output_dir'], 'models', ckpt_name)

    print(f'Training up to {config["epochs"]} epochs  (patience={config["patience"]})')
    for epoch in range(1, config['epochs'] + 1):
        model.train(); t0 = time.time(); tloss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tloss += loss.item() * len(xb)
        tloss /= len(x_t)

        model.eval()
        with torch.no_grad():
            vloss = criterion(model(x_v), y_v).item()

        scheduler.step(vloss)
        history['train_loss'].append(tloss); history['val_loss'].append(vloss)

        if vloss < best_val:
            best_val = vloss; patience_cnt = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_cnt += 1

        if epoch % 10 == 0 or epoch <= 5:
            lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch:4d}  train={tloss:.5f}  val={vloss:.5f}  '
                  f'best={best_val:.5f}  lr={lr:.2e}  ({time.time()-t0:.1f}s)')
        if patience_cnt >= config['patience']:
            print(f'Early stop at epoch {epoch}'); break

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'\nBest val={best_val:.6f}  loaded from {ckpt_path}')
    return model, history


print('Loss and training loop ready.')

## 10. Train

In [ ]:
model = TripleRegimeModel(n_outputs=4, dropout=CONFIG['dropout']).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
model, history = train_model(model, x_train, y_train, x_val, y_val, CONFIG)

## 11. Training curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ep = len(history['train_loss'])
ax.plot(range(1, ep+1), history['train_loss'], color=C_DL_NOISY, lw=1.5, label='Train')
ax.plot(range(1, ep+1), history['val_loss'],   color=C_TRIPLE,   lw=1.5, label='Val', ls='--')
ax.set_xlabel('Epoch', fontsize=10); ax.set_ylabel('Weighted MAE', fontsize=10)
ax.set_title('Triple-Regime (A+B+C) — Training Curve', fontsize=11, fontweight='bold')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, linestyle=':', alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout()
save_fig(fig, 'training_curve_triple')
plt.show()

with open(os.path.join(CONFIG['output_dir'], 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

## 12. Test-set evaluation

In [ ]:
y_pred_scaled = batched_predict(model, x_test)
y_pred_raw    = params_inverse(y_pred_scaled, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])

results = {}
print(f"\n{'='*70}")
print(f"{'Parameter':<10} {'RMSE':>10} {'Bias':>10} {'Pearson r':>12} {'R²':>8}")
print(f"{'-'*70}")
for i, (name, unit, sc) in enumerate(zip(PKEYS, PARAM_UNITS, PARAM_SCALE)):
    pred   = y_pred_raw[:, i] * sc
    true   = y_test_raw[:, i] * sc
    diff   = pred - true
    r      = float(np.corrcoef(true, pred)[0, 1])
    r2     = float(r2_score(true, pred))
    rms    = float(np.sqrt(np.mean(diff**2)))
    bias_v = float(np.mean(diff))
    results[name] = {'rmse': rms, 'bias': bias_v, 'r': r, 'r2': r2, 'unit': unit}
    print(f"{name:<10} {rms:>8.3f}{unit:>4} {bias_v:>+9.3f}{unit:>4} {r:>12.4f} {r2:>8.4f}")
print(f"{'='*70}")

with open(os.path.join(CONFIG['output_dir'], 'test_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

## 13. Figure — Scatter: predicted vs. ground truth
*(Same style as Fig 4 of 04_Generate_Figures_T2cond_v5.ipynb)*

In [ ]:
fig4, axes4 = plt.subplots(1, 4, figsize=(10, 2.8), gridspec_kw={'wspace': 0.38})

n_plot   = min(15000, len(y_pred_raw))
plot_idx = np.random.default_rng(42).choice(len(y_pred_raw), n_plot, replace=False)

for ci, (ax, pname, punit, pscale) in enumerate(zip(axes4, PARAM_NAMES, PARAM_UNITS, PARAM_SCALE)):
    t = y_test_raw[plot_idx, ci] * pscale
    p = y_pred_raw[plot_idx, ci] * pscale

    ax.scatter(t, p, s=1.5, alpha=0.10, color=C_TRIPLE, rasterized=True)
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.0, alpha=0.6)

    v = np.isfinite(t) & np.isfinite(p)
    if v.sum() > 10:
        r2_v = np.corrcoef(t[v], p[v])[0, 1] ** 2
        m, b = np.polyfit(t[v], p[v], 1)
        xf = np.array([lo, hi])
        ax.plot(xf, m*xf+b, color=C_DM, lw=1.5, alpha=0.9)
        rmse_v = np.sqrt(np.mean((t[v]-p[v])**2))
        ax.text(0.05, 0.93, f'R² = {r2_v:.3f}',
                transform=ax.transAxes, fontsize=8, fontweight='bold')
        ax.text(0.05, 0.83, f'RMSE = {rmse_v:.2f}',
                transform=ax.transAxes, fontsize=7.5, color='#333')

    ax.set_xlabel('True', fontsize=9)
    ax.set_ylabel('Pred', fontsize=9)
    ax.set_title(f'{pname} {punit}', fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig4.suptitle('Triple-Regime (A+B+C) — Predicted vs Ground Truth (Mixed SNR)',
              fontsize=10, y=1.02)
fig4.tight_layout()
save_fig(fig4, 'Fig_Scatter_TripleRegime')
plt.show()

## 14. Figure — RMSE per SNR
*(Same style as Fig 2B of 04_Generate_Figures_T2cond_v5.ipynb)*

In [ ]:
params_raw_all = load_mat(CONFIG['param_path'], CONFIG['param_key'])[:, :4]
per_snr = {name: [] for name in PKEYS}

for snr in SNR_LEVELS:
    print(f'  SNR={snr}...', end=' ')
    sig_path = os.path.join(CONFIG['dict_base_path'], f'QuasiRand_t2_snr{snr}.mat')
    sig_raw  = load_mat(sig_path, CONFIG['dict_key'])
    sig_i, par_i = filter_param_range(
        sig_raw, params_raw_all.copy(), CONFIG['param_mins'], CONFIG['param_maxs'])

    R2A, R2B, R2C = compute_triple_regime_features(sig_i, CONFIG)
    fA, fB, fC    = scale_triple_features(R2A, R2B, R2C, CONFIG)
    sig_norm, par_i = clean_data(euclidean_norm(sig_i), par_i)
    n = len(par_i)
    x_eval = np.concatenate([sig_norm, fA[:n,None], fB[:n,None], fC[:n,None]], axis=1).astype(np.float32)

    preds_raw = params_inverse(
        batched_predict(model, x_eval),
        CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])

    for j, (name, sc) in enumerate(zip(PKEYS, PARAM_SCALE)):
        per_snr[name].append(
            float(np.sqrt(np.mean((preds_raw[:, j]*sc - par_i[:, j]*sc)**2))))
    print(f'done  (N={n:,})')

# ── SNR line plot — matches reference style ──────────────────────────────────
x_pos = np.arange(len(SNR_LEVELS))

fig2b, axes2b = plt.subplots(1, 4, figsize=(12, 3.5), gridspec_kw={'wspace': 0.38})
fig2b.suptitle('Triple-Regime (A+B+C) — RMSE vs SNR',
               fontsize=10, fontweight='bold', x=0.02, ha='left')

for ax, (pkey, plabel, punit) in zip(axes2b, zip(PKEYS, PARAM_NAMES, PARAM_UNITS)):
    ax.plot(x_pos, per_snr[pkey],
            color=C_TRIPLE, marker='^', ls='-', lw=2, ms=6,
            label='Triple-regime (A+B+C)')

    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(s) for s in SNR_LEVELS], fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4); ax.set_axisbelow(True)
    if ax == axes2b[0]:
        ax.legend(fontsize=7.5, framealpha=0.9, loc='upper right')

fig2b.tight_layout()
save_fig(fig2b, 'Fig_RMSE_vs_SNR_TripleRegime')
plt.show()

# Save for later comparison
with open(os.path.join(CONFIG['output_dir'], 'per_snr_rmse.json'), 'w') as f:
    json.dump(per_snr, f, indent=2)

## 15. Ablation study — all three features

Each feature is set to 0.5 (neutral / uninformative) independently and in pairs,
matching the ablation format used in the dual-regime (A+B) notebook.

In [ ]:
# ── Reference results from dual-regime (A+B) notebook ────────────────────────
# Paste the exact numbers from your dual-regime ablation output here.
# Format: { condition_label: [SO2_%, CBV_%, R_um, T2_ms] }
DUAL_REGIME_ABLATION = {
    'Full (feat_A + feat_B)':    [13.477, 2.828, 5.285, 11.155],
    'Ablate feat_A (→0.5)':      [18.098, 3.200, 6.731, 12.701],
    'Ablate feat_B (→0.5)':      [13.779, 3.126, 7.455, 12.740],
    'Ablate both (→0.5)':        [16.810, 3.723, 6.857, 13.487],
}

# ── Triple-regime ablation conditions ────────────────────────────────────────
x_base = x_test.copy()

conditions_abc = {
    'Full (A+B+C)':               x_base.copy(),
    'Ablate feat_A (→0.5)':       x_base.copy(),
    'Ablate feat_B (→0.5)':       x_base.copy(),
    'Ablate feat_C (→0.5)':       x_base.copy(),
    'Ablate feat_A+B (→0.5)':     x_base.copy(),
    'Ablate feat_A+C (→0.5)':     x_base.copy(),
    'Ablate feat_B+C (→0.5)':     x_base.copy(),
    'Ablate all (→0.5)':          x_base.copy(),
}
conditions_abc['Ablate feat_A (→0.5)'][:,  40] = 0.5
conditions_abc['Ablate feat_B (→0.5)'][:,  41] = 0.5
conditions_abc['Ablate feat_C (→0.5)'][:,  42] = 0.5
conditions_abc['Ablate feat_A+B (→0.5)'][:, [40, 41]] = 0.5
conditions_abc['Ablate feat_A+C (→0.5)'][:, [40, 42]] = 0.5
conditions_abc['Ablate feat_B+C (→0.5)'][:, [41, 42]] = 0.5
conditions_abc['Ablate all (→0.5)'][:, [40, 41, 42]] = 0.5

# ── Run ablation ─────────────────────────────────────────────────────────────
ablation_results_abc = {}
header = f"{'Condition':<30}" + "".join(f"  {n+'('+u+')':>12}" for n, u in zip(PKEYS, PARAM_UNITS))
print('\nTriple-regime ablation study:\n')
print(header); print('-' * len(header))

for cond_name, x_cond in conditions_abc.items():
    preds_s = batched_predict(model, x_cond)
    preds_r = params_inverse(preds_s, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])
    row = []
    for j, sc in enumerate(PARAM_SCALE):
        rms = float(np.sqrt(np.mean((preds_r[:, j]*sc - y_test_raw[:, j]*sc)**2)))
        row.append(rms)
    ablation_results_abc[cond_name] = row
    print(f"{cond_name:<30}" + "".join(f"  {v:>12.3f}" for v in row))

with open(os.path.join(CONFIG['output_dir'], 'ablation_results.json'), 'w') as f:
    json.dump(ablation_results_abc, f, indent=2)

## 16. Figure — Ablation comparison: Dual (A+B) vs Triple (A+B+C)

Grouped bar chart comparing the ablation RMSE of both models side by side.

In [ ]:
# ── Map dual conditions to matching triple conditions ─────────────────────────
# We compare equivalent ablations: same feature(s) zeroed out.
compare_pairs = [
    ('Full (feat_A + feat_B)',  'Full (A+B+C)',           'Full'),
    ('Ablate feat_A (→0.5)',    'Ablate feat_A (→0.5)',   'No A'),
    ('Ablate feat_B (→0.5)',    'Ablate feat_B (→0.5)',   'No B'),
    ('Ablate both (→0.5)',      'Ablate feat_A+B (→0.5)', 'No A+B'),
]

param_info_abl = list(zip(PKEYS, PARAM_NAMES, PARAM_UNITS, PARAM_SCALE))
n_params  = len(param_info_abl)
n_conds   = len(compare_pairs)
bar_w     = 0.35
x_pos_abl = np.arange(n_conds)

fig_abl, axes_abl = plt.subplots(1, n_params, figsize=(12, 3.8),
                                   gridspec_kw={'wspace': 0.40})
fig_abl.suptitle('Ablation: Dual-Regime (A+B) vs Triple-Regime (A+B+C)',
                  fontsize=11, fontweight='bold')

for pi, (pkey, plabel, punit, pscale) in enumerate(param_info_abl):
    ax = axes_abl[pi]

    dual_vals   = [DUAL_REGIME_ABLATION[d][pi]          for d, _, _ in compare_pairs]
    triple_vals = [ablation_results_abc[t][pi]           for _, t, _ in compare_pairs]
    xlabels     = [lbl                                   for _, _, lbl in compare_pairs]

    bars1 = ax.bar(x_pos_abl - bar_w/2, dual_vals,   bar_w,
                   color=C_DL_NOISY, label='Dual (A+B)',   edgecolor='white', lw=0.5)
    bars2 = ax.bar(x_pos_abl + bar_w/2, triple_vals, bar_w,
                   color=C_TRIPLE,   label='Triple (A+B+C)', edgecolor='white', lw=0.5)

    # Value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.02*max(dual_vals+triple_vals),
                    f'{h:.2f}', ha='center', va='bottom', fontsize=6.5)

    ax.set_xticks(x_pos_abl)
    ax.set_xticklabels(xlabels, fontsize=8)
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4); ax.set_axisbelow(True)
    ymax = max(dual_vals + triple_vals)
    ax.set_ylim(0, ymax * 1.22)
    if pi == 0:
        ax.legend(fontsize=7.5, framealpha=0.9, loc='upper left')

fig_abl.tight_layout()
save_fig(fig_abl, 'Fig_Ablation_Dual_vs_Triple')
plt.show()

## 17. Figure — Comprehensive ablation heatmap (triple-regime only)

Shows all 8 ablation conditions as a colour-coded RMSE table.

In [ ]:
cond_labels = list(ablation_results_abc.keys())
matrix = np.array([ablation_results_abc[c] for c in cond_labels])  # (8, 4)

fig_hm, ax_hm = plt.subplots(figsize=(8, 4.5))
im = ax_hm.imshow(matrix, aspect='auto', cmap='YlOrRd')

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax_hm.text(j, i, f'{matrix[i,j]:.2f}',
                   ha='center', va='center', fontsize=8,
                   color='black' if matrix[i,j] < matrix.max()*0.75 else 'white')

ax_hm.set_xticks(range(n_params))
ax_hm.set_xticklabels([f'{n} {u}' for n, u in zip(PARAM_NAMES, PARAM_UNITS)], fontsize=9)
ax_hm.set_yticks(range(len(cond_labels)))
ax_hm.set_yticklabels(cond_labels, fontsize=8)
ax_hm.set_title('Triple-Regime Ablation — RMSE Heatmap',
                fontsize=11, fontweight='bold', pad=10)

cbar = fig_hm.colorbar(im, ax=ax_hm, fraction=0.035, pad=0.02)
cbar.set_label('RMSE (physical units)', fontsize=8)
cbar.ax.tick_params(labelsize=7)

plt.tight_layout()
save_fig(fig_hm, 'Fig_Ablation_Heatmap_Triple')
plt.show()

## 18. Save final model

In [ ]:
final_path = os.path.join(CONFIG['output_dir'], 'models', 'triple_regime_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'config':           {k: v.tolist() if isinstance(v, np.ndarray) else v
                         for k, v in CONFIG.items()},
    'results':          results,
    'per_snr_rmse':     per_snr,
    'ablation':         ablation_results_abc,
    'input_dim':        43,
    'input_layout':     '40 L2-norm echoes + feat_A + feat_B + feat_C',
    'feature_scaling':  {
        'R2starA': {'min': CONFIG['R2starA_min'], 'max': CONFIG['R2starA_max']},
        'R2starB': {'min': CONFIG['R2starB_min'], 'max': CONFIG['R2starB_max']},
        'R2starC': {'min': CONFIG['R2starC_min'], 'max': CONFIG['R2starC_max']},
    },
}, final_path)
print(f'Saved: {final_path}')
print()
print('Inference pipeline:')
print('  R2A, R2B, R2C = compute_triple_regime_features(sig_raw, CONFIG)')
print('  fA, fB, fC    = scale_triple_features(R2A, R2B, R2C, CONFIG)')
print('  x = np.concatenate([euclidean_norm(sig_raw), fA[:,None], fB[:,None], fC[:,None]], axis=1)')